<a href="https://colab.research.google.com/github/VexterLabs/SkyNet/blob/main/BrayanGPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 BrayanGPT: Entendiendo la IA desde adentro

¡Bienvenido a **BrayanGPT**! Este cuaderno es una **mini versión** de cómo funcionan los grandes cerebros artificiales como **ChatGPT** o **Gemini** por dentro.

Aquí vamos a construir, paso a paso, un modelo transformer capaz de generar texto. No es magia, es matemáticas y procesamiento de patrones.

### 🔗 ¡Sigue mi contenido en redes!
- 📺 **YouTube**: [@BrayanCode](https://www.youtube.com/@BrayanCode)
- 🐦 **X (Twitter)**: [x.com/BrayanCode](https://x.com/BrayanCode)
- 📸 **Instagram**: [instagram.com/brayancodemx](https://instagram.com/brayancodemx)
- 🎵 **TikTok**: [@brayancodemx](https://tiktok.com/@brayancodemx)

---

### 1. Las herramientas básicas (Importaciones)

Primero, traemos las herramientas. `torch` es como la "caja de herramientas" que nos permite crear neuronas artificiales y hacer cálculos rápidos.

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import time
import sys

torch.manual_seed(1337)

### 2. Configurando el "Tamaño del Cerebro"

Decidimos qué tan inteligente y profundo será nuestro **BrayanGPT**.
- `block_size`: ¿Cuántas letras puede recordar a la vez? (Su memoria de corto plazo).
- `n_layer`: ¿Cuántas capas de "pensamiento" tiene? (Su profundidad).

In [ ]:
class Config:
    n_embd = 192
    n_head = 6
    n_layer = 6
    batch_size = 64
    block_size = 128
    max_iters = 3000
    learning_rate = 1e-3
    dropout = 0.25
    eval_interval = 250
    eval_iters = 100
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Sistema listo. 'Mexicanizador' operando en: {Config.device}")


Configuración lista para el Mexicanizador en: cuda


### 3. El Lenguaje de la IA: Tokenización

Las computadoras no entienden palabras, entienden **números**.

En **BrayanGPT**, cada letra o símbolo es un número único. Veamos cómo se ve este proceso visualmente:

In [ ]:
import gdown
import os

# Descargar dataset 'Mexicanizador.txt'
file_id = '11HLaXR9otMOxQisTSi9cDFI2vCwtAC9j'
url = f'https://drive.google.com/uc?id={file_id}'
output_file = 'Mexicanizador.txt'

if not os.path.exists(output_file):
    print("Descargando 'Mexicanizador.txt' desde Google Drive...")
    gdown.download(url, output_file, quiet=False)
else:
    print("El archivo ya se encuentra en el entorno de Colab.")

try:
    with open('Mexicanizador.txt', 'r', encoding='utf-8') as f:
        raw_text = f.read()
except FileNotFoundError:
    raw_text = "¡Hola! Soy BrayanGPT y estoy aprendiendo a escribir como un humano... " * 500

chars = sorted(list(set(raw_text)))
vocab_size = len(chars)

char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [char_to_idx[c] for c in s]
decode = lambda l: ''.join([idx_to_char[i] for i in l])

print(f"\n--- Detalles del Vocabulario ---")
print(f"Cantidad de letras únicas que conozco: {vocab_size}")
print(f"Total de letras en mi libro de estudio: {len(raw_text)}")
print(f"Letras que domino: {''.join(chars[:20])}...")

print(f"\n--- Ejemplo Visual de Tokenización ---")
frase_ejemplo = "BrayanCode"
tokens = encode(frase_ejemplo)

print(f"Frase Humana: '{frase_ejemplo}'")
for char, token in zip(frase_ejemplo, tokens):
    print(f" Letra '{char}' -> Número {token}")

print(f"\nRepresentación final para la IA: {tokens}")

--- Detalles del Vocabulario ---
Cantidad de letras únicas que conozco: 114
Total de letras en mi libro de estudio: 1013765
Letras que domino: 
 !"$%'()*,-./012345...

--- Ejemplo Visual de Tokenización ---
Frase Humana: 'BrayanCode'
 Letra 'B' -> Número 28
 Letra 'r' -> Número 72
 Letra 'a' -> Número 55
 Letra 'y' -> Número 79
 Letra 'a' -> Número 55
 Letra 'n' -> Número 68
 Letra 'C' -> Número 29
 Letra 'o' -> Número 69
 Letra 'd' -> Número 58
 Letra 'e' -> Número 59

Representación final para la IA: [28, 72, 55, 79, 55, 68, 29, 69, 58, 59]


### 4. Dividiendo el Conocimiento

Separamos el texto: 90% para aprender (**Entrenamiento**) y 10% para evaluarnos (**Validación**).

In [ ]:
full_data = torch.tensor(encode(raw_text), dtype=torch.long)
n_split = int(0.9 * len(full_data))
train_data, val_data = full_data[:n_split], full_data[n_split:]

def get_batch(split_type):
    data_src = train_data if split_type == 'train' else val_data
    ix = torch.randint(len(data_src) - Config.block_size, (Config.batch_size,))
    x = torch.stack([data_src[i:i+Config.block_size] for i in ix])
    y = torch.stack([data_src[i+1:i+Config.block_size+1] for i in ix])
    return x.to(Config.device), y.to(Config.device)

### 5. ¿Cómo "Mira" la IA? (Atención Mecánica)

La **Atención** le permite a la IA saber qué letras anteriores son más importantes para decidir cuál sigue. Es como tener varios filtros que resaltan lo relevante.

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(Config.n_embd, head_size, bias=False)
        self.query = nn.Linear(Config.n_embd, head_size, bias=False)
        self.value = nn.Linear(Config.n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(Config.block_size, Config.block_size)))
        self.dropout = nn.Dropout(Config.dropout)

    def forward(self, x):
        B, T, C = x.shape
        k, q, v = self.key(x), self.query(x), self.value(x)
        wei = q @ k.transpose(-2, -1) * (C**-0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        return self.dropout(wei) @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([CausalSelfAttention(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(Config.n_embd, Config.n_embd)
        self.dropout = nn.Dropout(Config.dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))

### 6. El Bloque Transformer (Pensar y Procesar)

Aquí unimos la **Atención** (comunicación) con una **Red Neuronal** (reflexión). Esta estructura repetida varias veces es lo que hace que ChatGPT sea tan potente.

In [ ]:
class MLP(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(Config.dropout),
        )
    def forward(self, x):
        return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = MLP(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

### 7. Ensamblando la BrayanGPT

Este es el diseño final. Conecta los tokens con el cerebro transformer.

In [ ]:
class BrayanGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, Config.n_embd)
        self.pos_emb = nn.Embedding(Config.block_size, Config.n_embd)
        self.blocks = nn.Sequential(*[
            TransformerBlock(Config.n_embd, Config.n_head) for _ in range(Config.n_layer)
        ])
        self.ln_final = nn.LayerNorm(Config.n_embd)
        self.lm_head = nn.Linear(Config.n_embd, vocab_size)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None: torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_e = self.token_emb(idx)
        pos_e = self.pos_emb(torch.arange(T, device=Config.device))
        x = tok_e + pos_e
        x = self.blocks(x)
        x = self.ln_final(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B*T, C), targets.view(B*T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -Config.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
            yield idx_next

### 8. ¡Hora de estudiar! (Entrenamiento)

Aquí la IA leerá el texto miles de veces ajustando sus neuronas. Al principio dirá letras al azar, pero poco a poco ganará coherencia.

In [ ]:
model = BrayanGPT().to(Config.device)
optimizer = torch.optim.AdamW(model.parameters(), lr=Config.learning_rate)
print(f"Cerebro de BrayanGPT creado con {sum(p.numel() for p in model.parameters())/1e6:.2f}M de parámetros.")

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(Config.eval_iters)
        for k in range(Config.eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

print("--- Iniciando estudio intenso ---")
for iter in range(Config.max_iters):
    if iter % Config.eval_interval == 0 or iter == Config.max_iters - 1:
        losses = estimate_loss()
        print(f"Pasos: {iter} | Error Entrenamiento: {losses['train']:.4f} | Error Validación: {losses['val']:.4f}")

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

Cerebro de BrayanGPT creado con 2.73M de parámetros.
--- Iniciando estudio intenso ---
Pasos: 0 | Error Entrenamiento: 4.6828 | Error Validación: 4.6865
Pasos: 250 | Error Entrenamiento: 2.3862 | Error Validación: 2.3929
Pasos: 500 | Error Entrenamiento: 2.1518 | Error Validación: 2.1619
Pasos: 750 | Error Entrenamiento: 1.8628 | Error Validación: 1.8964
Pasos: 1000 | Error Entrenamiento: 1.6534 | Error Validación: 1.6922
Pasos: 1250 | Error Entrenamiento: 1.4870 | Error Validación: 1.5501
Pasos: 1500 | Error Entrenamiento: 1.3826 | Error Validación: 1.4546
Pasos: 1750 | Error Entrenamiento: 1.3117 | Error Validación: 1.4086
Pasos: 2000 | Error Entrenamiento: 1.2503 | Error Validación: 1.3617
Pasos: 2250 | Error Entrenamiento: 1.2147 | Error Validación: 1.3280
Pasos: 2500 | Error Entrenamiento: 1.1810 | Error Validación: 1.3095
Pasos: 2750 | Error Entrenamiento: 1.1537 | Error Validación: 1.2955
Pasos: 2999 | Error Entrenamiento: 1.1292 | Error Validación: 1.2781


### 9. ¡A escribir! (Generación)

Finalmente, le pedimos a nuestra IA que genere texto de la nada.

In [ ]:
print("\n=== RESULTADO DE BRAYANGPT ===")
contexto_inicial = torch.tensor([encode('\n')], dtype=torch.long, device=Config.device)
model.eval()

for token_tensor in model.generate(contexto_inicial, max_new_tokens=500, temperature=0.7, top_k=40):
    char = decode(token_tensor[0].tolist())
    print(char, end='')
    sys.stdout.flush()
    time.sleep(0.01)

print("\n\n=== Fin de la generación ===")


=== RESULTADO DE BRAYANGPT ===
Lo peor es que el día no es como si fuera su oficina. Alguien sudando en la banqueta, ese cabrón más tequila que iba a rezar a la casa de las Noas, y me voy a decir que me voy a dice: "¡Órale, así está bien pendejo!" ¡Y el calorón de adobada sudado, wey! ¡No mames! Caminando el billete bien filte con las carne tortillas en el primer de la calle, de la noche, no mames. El Toño me dijo: "¡Puro cabezal! ¿Que está en la calle a la papel de la gorda de la carne asada, eh?".

El mero me dijo: "¡Soy de

=== Fin de la generación ===
